# Query Functions for At Bat Strategizer

This notebook creates Unity Catalog functions and vector search indices to support the At Bat Strategizer agent.

## Tooling Overview

1. **Batter-Pitcher Matchup**: Get all pitches faced by specific batter-pitcher combination
2. **Pitcher Tendencies**: Get pitch distribution by count, handedness, and location
3. **Similar Pitchers**: Find similar pitcher-pitch types using vector search
4. **Similar Batters**: Find similar batters using vector search
5. **Scenario Queries**: Query pitches by count and game situation


In [ ]:
# Install required packages
%pip install databricks-connect databricks-vectorsearch
%restart_python

In [0]:
# Load configuration from setup notebook
import json
from pathlib import Path

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

catalog = CONFIG["workspace"]["catalog"]
schema = CONFIG["workspace"]["schema"]
vector_search_endpoint = CONFIG["vector_search"]["endpoint_name"]

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Vector Search Endpoint: {vector_search_endpoint}")


## Step 1: Enable Change Data Feed on Source Tables

Vector search indices require Change Data Feed (CDF) to be enabled on the source Delta tables.

### Enable CDF for pitcher_vectors_mean


In [ ]:
# Enable Change Data Feed for pitcher_vectors_mean table
spark.sql(f"""
    ALTER TABLE {catalog}.{schema}.pitcher_vectors_mean 
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print(f"✓ Enabled Change Data Feed for {catalog}.{schema}.pitcher_vectors_mean")


### Enable CDF for batter_vectors_mean


In [ ]:
# Enable Change Data Feed for batter_vectors_mean table
spark.sql(f"""
    ALTER TABLE {catalog}.{schema}.batter_vectors_mean 
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print(f"✓ Enabled Change Data Feed for {catalog}.{schema}.batter_vectors_mean")


## Step 2: Create Vector Search Indices

Now that CDF is enabled, we can create the vector search indices.


In [ ]:
# Create vector search index for pitcher_vectors_mean
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

pitcher_index_name = f"{catalog}.{schema}.pitcher_vectors_mean_index"
pitcher_source_table = f"{catalog}.{schema}.pitcher_vectors_mean"

try:
    # Create the index
    vsc.create_delta_sync_index(
        endpoint_name=vector_search_endpoint,
        index_name=pitcher_index_name,
        source_table_name=pitcher_source_table,
        pipeline_type="TRIGGERED",
        primary_key="player_id",
        embedding_dimension=16,  # 16 features in pitcher vectors
        embedding_vector_column="embedding_vector"
    )
    print(f"✓ Created pitcher vector search index: {pitcher_index_name}")
except Exception as e:
    if "already exists" in str(e):
        print(f"✓ Pitcher vector search index already exists: {pitcher_index_name}")
    else:
        print(f"Error creating pitcher index: {e}")

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Create vector search index for batter_vectors_mean
batter_index_name = f"{catalog}.{schema}.batter_vectors_mean_index"
batter_source_table = f"{catalog}.{schema}.batter_vectors_mean"

try:
    # Create the index
    vsc.create_delta_sync_index(
        endpoint_name=vector_search_endpoint,
        index_name=batter_index_name,
        source_table_name=batter_source_table,
        pipeline_type="TRIGGERED",
        primary_key="player_id",
        embedding_dimension=10,  # 10 features in batter vectors
        embedding_vector_column="embedding_vector"
    )
    print(f"✓ Created batter vector search index: {batter_index_name}")
except Exception as e:
    if "already exists" in str(e):
        print(f"✓ Batter vector search index already exists: {batter_index_name}")
    else:
        print(f"Error creating batter index: {e}")


## Step 3: Create and Configure Genie Space

Before proceeding to notebook 03 (agent definition), you must create a **Genie Space** in the workspace UI and update the secret scope with its ID.

### Create the Genie Space

1. In the Databricks workspace, navigate to **Genie** → **New Space**
2. Name it something like `At-Bat Assistant Baseball Data`
3. Add the following tables from `{catalog}.{schema}`:
   - `statcast_pitches`
   - `batter_vectors_median`, `batter_vectors_mean`
   - `dim_batter_team_year`, `dim_batters`
   - `dim_pitchers`, `dim_pitcher_arsenal`, `dim_pitcher_team_year`
   - `dim_players`
   - `pitcher_vectors_median`, `pitcher_vectors_mean`
4. Copy the **Genie Space ID** from the URL (it's the UUID in the URL path, e.g. `https://<workspace>/genie/rooms/<GENIE_SPACE_ID>`)

### Update the Secret Scope

After creating the Genie Space, update the secret scope so notebook 00 picks up the real ID when regenerating the config:

```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.secrets.put_secret(scope="atbat-assistant-secrets", key="genie-space-id", string_value="<your Genie Space ID>")
```

Then **re-run notebook 00** to regenerate `config/atbat_assistant.json` with the real Genie Space ID. The agent in notebook 03 will fail with a `PermissionDenied: You need "Can View" permission` error if the Genie Space ID is missing or invalid.

## Step 2: Create Unity Catalog Functions

These functions will be available as tools for AI agents.


In [ ]:
### Function 1: lookup_player_by_name
# Find player ID by name (searches pitchers only since batters don't have names in Statcast data)

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.lookup_player_by_name(
name_f STRING COMMENT 'The first name of the player to look up (e.g., "shohei")', 
name_l STRING COMMENT 'The last name of the player to look up (e.g., "ohtani")'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Look up player ID by name. Searches pitchers and batters. name_first is one column, name_last is another column'
RETURN (
  SELECT *
  FROM {catalog}.{schema}.dim_players 
  WHERE LOWER(name_first) = LOWER(name_f)
    AND LOWER(name_last) = LOWER(name_l)
  LIMIT 1
)
""")

print(f"✓ Created function: {catalog}.{schema}.lookup_player_by_name")


In [ ]:
### Function 2: get_batter_pitcher_matchup
# Retrieve all pitches faced by specific batter-pitcher combination

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_batter_pitcher_matchup(
  b_id INT COMMENT 'The batter ID to look up',
  p_id INT COMMENT 'The pitcher ID to look up',
  season_year INT COMMENT 'The season year to look up'
)
RETURNS TABLE(
  game_date DATE,
  balls INT,
  strikes INT,
  pitch_type STRING,
  pitch_name STRING,
  release_speed DOUBLE,
  release_spin_rate DOUBLE,
  plate_x DOUBLE,
  plate_z DOUBLE,
  pfx_x DOUBLE,
  pfx_z DOUBLE,
  events STRING,
  description STRING,
  launch_speed DOUBLE,
  launch_angle DOUBLE,
  hit_distance_sc DOUBLE
)
LANGUAGE SQL
COMMENT 'Get all pitches faced by a specific batter-pitcher combination in a season'
RETURN (
  SELECT 
    game_date,
    balls,
    strikes,
    pitch_type,
    pitch_name,
    release_speed,
    release_spin_rate,
    plate_x,
    plate_z,
    pfx_x,
    pfx_z,
    events,
    description,
    launch_speed,
    launch_angle,
    hit_distance_sc
  FROM {catalog}.{schema}.statcast_pitches
  WHERE batter = b_id
    AND pitcher = p_id
    AND season = season_year
  ORDER BY game_date, at_bat_number, pitch_number
)
""")

print(f"✓ Created function: {catalog}.{schema}.get_batter_pitcher_matchup")


In [ ]:
### Function 3: get_pitcher_tendency_by_count
# Get pitch distribution by count with location and sorted by frequency

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_pitcher_tendency_by_count(
  p_id INT COMMENT 'The pitcher ID to look up',
  b_hand STRING COMMENT 'The batter hand to look up',
  b INT COMMENT 'The number of balls in the count',
  s INT COMMENT 'The number of strikes in the count',
  season_year INT COMMENT 'The season year to look up'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get pitcher tendency by count: pitch type, location zone, and frequency. Returns JSON sorted by pitch type then frequency.'
RETURN (
  WITH pitch_locations AS (
    SELECT
      pitch_type,
      pitch_name,
      CASE
        WHEN plate_x < -0.33 AND plate_z > 2.5 THEN 'high_away'
        WHEN plate_x BETWEEN -0.33 AND 0.33 AND plate_z > 2.5 THEN 'high_middle'
        WHEN plate_x > 0.33 AND plate_z > 2.5 THEN 'high_in'
        WHEN plate_x < -0.33 AND plate_z BETWEEN 1.5 AND 2.5 THEN 'middle_away'
        WHEN plate_x BETWEEN -0.33 AND 0.33 AND plate_z BETWEEN 1.5 AND 2.5 THEN 'middle_middle'
        WHEN plate_x > 0.33 AND plate_z BETWEEN 1.5 AND 2.5 THEN 'middle_in'
        WHEN plate_x < -0.33 AND plate_z < 1.5 THEN 'low_away'
        WHEN plate_x BETWEEN -0.33 AND 0.33 AND plate_z < 1.5 THEN 'low_middle'
        WHEN plate_x > 0.33 AND plate_z < 1.5 THEN 'low_in'
        ELSE 'unknown'
      END as location_zone,
      COUNT(*) as pitch_count
    FROM {catalog}.{schema}.statcast_pitches
    WHERE pitcher = p_id
      AND stand = b_hand
      AND balls = b
      AND strikes = s
      AND season = season_year
      AND plate_x IS NOT NULL
      AND plate_z IS NOT NULL
    GROUP BY pitch_type, pitch_name, location_zone
  ),
  pitch_data_with_frequency AS (
    SELECT
      pl.pitch_type,
      pl.pitch_name,
      pl.location_zone,
      pl.pitch_count,
      ROUND(100.0 * pl.pitch_count / pt.total_pitches, 1) as frequency_pct
    FROM pitch_locations pl
    CROSS JOIN (SELECT SUM(pitch_count) as total_pitches FROM pitch_locations) pt
  ),
  -- New CTE to create the struct and apply the ORDER BY via ROW_NUMBER for guaranteed sort
  sorted_pitch_data AS (
      SELECT
          struct(
              pitch_type,
              pitch_name,
              location_zone,
              pitch_count,
              frequency_pct
          ) AS pitch_struct
      FROM pitch_data_with_frequency
      ORDER BY pitch_type, frequency_pct DESC
  )
  SELECT to_json(collect_list(pitch_struct))
  FROM sorted_pitch_data
)
""")

print(f"✓ Created function: {catalog}.{schema}.get_pitcher_tendency_by_count")


In [ ]:
### Function 4: get_pitcher_tendency_with_runners
# Get pitch distribution by count AND base runner situation

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_pitcher_tendency_with_runners(
  p_id INT COMMENT 'The pitcher ID to look up',
  b_hand STRING COMMENT 'The batter hand to look up',
  b INT COMMENT 'The number of balls in the count. If the desired count has 0 balls, you must provide 0 rather than nothing',
  s INT COMMENT 'The number of strikes in the count If the desired count has 0 balls, you must provide 0 rather than nothing',
  p_on_1b BOOLEAN COMMENT 'Whether there is a runner on first base',
  p_on_2b BOOLEAN COMMENT 'Whether there is a runner on second base',
  p_on_3b BOOLEAN COMMENT 'Whether there is a runner on third base',
  season_year INT COMMENT 'The season year to look up'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Get pitcher tendency by count and base runner situation. Returns JSON with pitch type, location, and frequency.'
RETURN (
  WITH pitch_locations AS (
    SELECT 
      pitch_type,
      pitch_name,
      CASE 
        WHEN plate_x < -0.33 AND plate_z > 2.5 THEN 'high_away'
        WHEN plate_x BETWEEN -0.33 AND 0.33 AND plate_z > 2.5 THEN 'high_middle'
        WHEN plate_x > 0.33 AND plate_z > 2.5 THEN 'high_in'
        WHEN plate_x < -0.33 AND plate_z BETWEEN 1.5 AND 2.5 THEN 'middle_away'
        WHEN plate_x BETWEEN -0.33 AND 0.33 AND plate_z BETWEEN 1.5 AND 2.5 THEN 'middle_middle'
        WHEN plate_x > 0.33 AND plate_z BETWEEN 1.5 AND 2.5 THEN 'middle_in'
        WHEN plate_x < -0.33 AND plate_z < 1.5 THEN 'low_away'
        WHEN plate_x BETWEEN -0.33 AND 0.33 AND plate_z < 1.5 THEN 'low_middle'
        WHEN plate_x > 0.33 AND plate_z < 1.5 THEN 'low_in'
        ELSE 'unknown'
      END as location_zone,
      COUNT(*) as pitch_count
    FROM {catalog}.{schema}.statcast_pitches
    WHERE pitcher = p_id
      AND stand = b_hand
      AND balls = b
      AND strikes = s
      AND season = season_year
      AND plate_x IS NOT NULL
      AND plate_z IS NOT NULL
      
      -- NEW, CLEANER RUNNER LOGIC: Compare BOOLEAN parameter to the column's IS NULL status.
      AND (p_on_1b IS NULL OR (on_1b IS NOT NULL) = p_on_1b) 
      AND (p_on_2b IS NULL OR (on_2b IS NOT NULL) = p_on_2b)
      AND (p_on_3b IS NULL OR (on_3b IS NOT NULL) = p_on_3b)
      
    GROUP BY pitch_type, pitch_name, location_zone
  ),
  pitch_data_with_frequency AS (
    SELECT
      pl.pitch_type,
      pl.pitch_name,
      pl.location_zone,
      pl.pitch_count,
      ROUND(100.0 * pl.pitch_count / pt.total_pitches, 1) as frequency_pct
    FROM pitch_locations pl
    CROSS JOIN (SELECT SUM(pitch_count) as total_pitches FROM pitch_locations) pt
  ),
  sorted_pitch_data AS (
      SELECT
          struct(
              pitch_type,
              pitch_name,
              location_zone,
              pitch_count,
              frequency_pct
          ) AS pitch_struct
      FROM pitch_data_with_frequency
      ORDER BY pitch_type, frequency_pct DESC
  )
  SELECT to_json(collect_list(pitch_struct))
  FROM sorted_pitch_data
)
""")
print(f"✓ Created function: {catalog}.{schema}.get_pitcher_tendency_with_runners")


In [ ]:
### Function 5: Get embedding for a pitcher

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.pitcher_embedding_lookup(
p_id BIGINT COMMENT 'The pitcher ID to look up', 
season_year BIGINT COMMENT 'The season year to look up', 
p_type STRING COMMENT 'The pitch type to look up (ex. FF, SL, CH, CU, etc.)'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Look up player ID by ID, season and pitch type. Returns the embedding vector for the pitcher which is used for pitcher vector search. Pitcher arsenal should be run first to get all the pitches a pitcher throws'
RETURN (
  SELECT player_id, pitch_type, embedding_vector
  FROM {catalog}.{schema}.pitcher_vectors_mean 
  WHERE player_id = p_id
    AND season = season_year
    AND pitch_type = p_type
  LIMIT 1
)
""")


In [ ]:
### Function 6: Get pitcher arsenal 

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.pitcher_arsenal_lookup(
p_id BIGINT COMMENT 'The pitcher ID to look up'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Look up player ID by ID and pitch type. Returns all the pitches this pitcher throws.'
RETURN (
  SELECT player_id, pitch_type
  FROM {catalog}.{schema}.dim_pitcher_arsenal
  WHERE player_id = p_id
)
""")

In [ ]:
### Function 7: Get batter embedding vector
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.batter_embedding_lookup(
p_id BIGINT COMMENT 'The batter ID to look up', 
season_year BIGINT COMMENT 'The season year to look up', 
p_type STRING COMMENT 'The pitch type to look up (ex. FF, SL, CH, CU, etc.)'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Look up player ID by ID, season and pitch type. Returns the embedding vector for the batter which is used for batter vector search.'
RETURN (
  SELECT player_id, pitch_type, embedding_vector
  FROM {catalog}.{schema}.batter_vectors_mean 
  WHERE player_id = p_id
    AND season = season_year
    AND pitch_type = p_type
  LIMIT 1
)
""")

In [0]:
batter_index_name = CONFIG["vector_search"]["batter_index_name"]

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.batter_embedding_query(e_vector ARRAY<FLOAT> COMMENT 'The embedding vector (list of floats) to search for similar batters in the index')
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Query Vector Search index for similar batter - pitch type combinations'
RETURN (
SELECT * FROM vector_search(
  index =>"{batter_index_name}",
  query_vector => e_vector,
  num_results => 6,
  query_type => 'ANN'
)
)
""")
print(f"Created function: {catalog}.{schema}.batter_embedding_query")


In [0]:
pitcher_index_name = CONFIG["vector_search"]["pitcher_index_name"]

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.pitcher_embedding_query(e_vector ARRAY<FLOAT> COMMENT 'The embedding vector (list of floats) to search for similar pitchers in the index')
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Query Vector Search index for similar pitcher - pitch type combinations'
RETURN (
SELECT * FROM vector_search(
  index =>"{pitcher_index_name}",
  query_vector => e_vector,
  num_results => 6,
  query_type => 'ANN'
)
)
""")
print(f"Created function: {catalog}.{schema}.pitcher_embedding_query")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_team_batters(
  team_abbr STRING COMMENT 'team abbreviation used for the query',
  season_year INT COMMENT 'season year for the query'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Return full batter roster for a team and season using dim_batter_team_year. Use this for requests that want to understand which batters will perform well vs. a specific pitcher to understand all the batters on a specific team.'
RETURN (
  SELECT player_id, player_name, team
  FROM {catalog}.{schema}.dim_batter_team_year
  WHERE year = season_year AND team = team_abbr
  ORDER BY player_name
)
""")
print(f"✓ Created function: {catalog}.{schema}.get_team_batters")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.recommend_batter_matchups_by_team(
  p_id BIGINT COMMENT 'pitcher id for the query',
  team_abbr STRING COMMENT 'team abbreviation used for the query',
  season_year INT COMMENT 'season year for the query'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Weighted expected wOBA per roster batter using pitcher usage by batter-hand and batter performance vs pitch types. Use this for requests that want to understand which batters will perform well vs. a specific pitcher'
RETURN (
  WITH p_hand AS (
    SELECT COALESCE(throws, 'R') AS p_throws
    FROM {catalog}.{schema}.dim_pitchers
    WHERE player_id = p_id
    LIMIT 1
  ),
  roster AS (
    SELECT r.player_id,
           r.player_name,
           r.team,
           dp.bats,
           CASE
             WHEN dp.bats = 'S' THEN CASE (SELECT p_throws FROM p_hand) WHEN 'R' THEN 'L' WHEN 'L' THEN 'R' ELSE 'R' END
             ELSE dp.bats
           END AS eff_stand
    FROM {catalog}.{schema}.dim_batter_team_year r
    LEFT JOIN {catalog}.{schema}.dim_players dp ON r.player_id = dp.player_id
    WHERE r.year = season_year AND r.team = team_abbr
  ),
  usage_base AS (
    SELECT pitch_type, stand, COUNT(*) AS cnt
    FROM {catalog}.{schema}.statcast_pitches
    WHERE pitcher = p_id
      AND season = season_year
      AND pitch_type IS NOT NULL
      AND stand IN ('L','R')
    GROUP BY pitch_type, stand
  ),
  usage_by_hand AS (
    SELECT pitch_type,
           stand,
           cnt / NULLIF(SUM(cnt) OVER (PARTITION BY stand), 0) AS usage_share
    FROM usage_base
  ),
  batter_perf AS (
    SELECT player_id,
           pitch_type,
           stand,
           estimated_woba_using_speedangle AS perf
    FROM {catalog}.{schema}.batter_vectors_mean
    WHERE season = season_year
  ),
  scored AS (
    SELECT
      r.player_id,
      r.player_name,
      r.team,
      SUM(bp.perf * ubh.usage_share) AS expected_woba
    FROM roster r
    JOIN batter_perf bp
      ON bp.player_id = r.player_id
     AND bp.stand = r.eff_stand
    JOIN usage_by_hand ubh
      ON ubh.pitch_type = bp.pitch_type
     AND ubh.stand = r.eff_stand
    GROUP BY r.player_id, r.player_name, r.team
  )
  SELECT *
  FROM scored
  ORDER BY expected_woba DESC
  LIMIT 30
)
""")
print(f"✓ Created function: {catalog}.{schema}.recommend_batter_matchups_by_team")

In [ ]:
# Batch embeddings lookup for many batters at once (optional pitch type filter)
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.batter_embeddings_for_ids(
  b_ids ARRAY<BIGINT> COMMENT 'An array of batter ids to lookup',
  season_year INT COMMENT 'The season to lookup'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Returns embeddings from batter_vectors_mean for many batters. This should be executed after recommend_batter_matchups_by_team is executed to understand similarities of batters that have good expected outcomes vs. a specific pitcher. The table will return all pitch types.'
RETURN (
  WITH ids AS (SELECT explode(b_ids) AS player_id)
  SELECT
    b.player_id,
    dp.name_first,
    dp.name_last,
    b.season,
    b.pitch_type,
    b.stand,
    b.embedding_vector
  FROM {catalog}.{schema}.batter_vectors_mean b
  JOIN ids i ON b.player_id = i.player_id
  LEFT JOIN {catalog}.{schema}.dim_players dp ON dp.player_id = b.player_id
  WHERE b.season = season_year
  ORDER BY b.player_id, b.season, b.pitch_type
)
""")
print(f"✓ Created function: {catalog}.{schema}.batter_embeddings_for_ids")